In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import gradio as gr
import os
import tempfile
import uuid
from scipy.stats import norm, chi2

In [ ]:
def norm_cdf(x):
  return norm.cdf(x, loc=0, scale=1)

def p_value_from_z(z):
  return 2 * (1 - norm_cdf(abs(z)))

def z_critical(confidence_level):
  alpha = 1 - confidence_level
  return norm.ppf(1 - alpha / 2)

def z_alpha(conf_level):
    alpha = 1.0 - conf_level
    return norm.ppf(1 - alpha/2)

def z_power(power):
    return norm.ppf(power)

In [ ]:
def mdu_from_baseline(p_base, n_c, n_v, conf_level, power):
    if n_c <= 0 or n_v <= 0:
        return float("nan"), float("nan")
    if p_base <= 0:
        return float("nan"), float("nan")

    zc = z_alpha(conf_level)
    zp = z_power(power)

    se0 = math.sqrt(p_base * (1 - p_base) / n_c + p_base * (1 - p_base) / n_v)
    mdu_abs = (zc + zp) * se0
    mdu_rel = mdu_abs / p_base
    return mdu_abs, mdu_rel

In [ ]:
def chi_square_p_value_2x2(x_c, x_v, n_c, n_v):
  fail_c = n_c - x_c
  fail_v = n_v - x_v

  obs = [[x_c, fail_c],[x_v, fail_v]]

  row_tot = [sum(obs[0]), sum(obs[1])]
  col_tot = [obs[0][0] + obs[1][0], obs[0][1] + obs[1][1]]
  grand = sum(row_tot)

  exp = [[row_tot[0] * col_tot[0] / grand, row_tot[0] * col_tot[1] / grand],[row_tot[1] * col_tot[0] / grand, row_tot[1] * col_tot[1] / grand]]

  # Chi-square statistic for a 2x2 contingency table
  chi2_stat = (
      (obs[0][0] - exp[0][0])**2 / exp[0][0] +
      (obs[0][1] - exp[0][1])**2 / exp[0][1] +
      (obs[1][0] - exp[1][0])**2 / exp[1][0] +
      (obs[1][1] - exp[1][1])**2 / exp[1][1]
  )
  p_value = 1.0 - chi2.cdf(chi2_stat, df=1)

  return chi2_stat, p_value

In [ ]:
def compute(df_tests):

  required = {"test_id", "source", "definition", "from_to", "n_control", "x_control", "n_variant", "x_variant", "conf_level"}
  missing = required - set(df_tests.columns)

  if missing:
      raise ValueError(f"Faltan columnas en df_tests: {sorted(missing)}")

  df = df_tests.copy()

  summary_rows = []
  detail_rows = []

  for index, row in df.iterrows():

    test_id = row["test_id"]
    source = row["source"]
    definition = row["definition"]
    from_to = row["from_to"]
    role = row["role"] if "role" in df.columns else None
    metric_name = row["metric_name"] if "metric_name" in df.columns else None
    variant_id = row["variant_id"] if "variant_id" in df.columns else None
    n_c = row["n_control"]
    x_c = row["x_control"]
    n_v = row["n_variant"]
    x_v = row["x_variant"]
    conf_level = float(row["conf_level"])
    mde_expected = float(row["mde_expected"]) if "mde_expected" in df.columns else None


    if "power" in df.columns and not pd.isna(row["power"]):
      power = float(row["power"])
    else:
      power = 0.8

    zcrit = z_critical(conf_level)  # Valor crítico z: “multiplicador” de la normal estándar para construir el intervalo de confianza.

    p_c = x_c / n_c  # Tasa de conversión del grupo de control
    p_v = x_v / n_v  # Tasa de conversión del grupo variante

    se_c = math.sqrt(p_c * (1.0 - p_c) / n_c)  # Error estándar de las tasas
    se_v = math.sqrt(p_v * (1.0 - p_v) / n_v)

    ci_c_low, ci_c_high = p_c - zcrit * se_c, p_c + zcrit * se_c  # Intervalos de confianza del rate: Si repitiera el experimento muchas veces, ~95% de los IC incluirían la tasa real
    ci_v_low, ci_v_high = p_v - zcrit * se_v, p_v + zcrit * se_v

    if p_c == 0:
      lift = float("inf")
      se_lift = float("nan")
      z_lift = float("nan")
      p_value = float("nan")
      confidence = float("nan")
      ci_lift_low = float("nan")
      ci_lift_high = float("nan")

    else:
      lift = (p_v - p_c)/ p_c  # Lift (efecto relativo): Mejora relativa de Variante frente a Control.
      se_lift = math.sqrt((se_v / p_c) ** 2 + ((p_v * se_c) / (p_c ** 2)) ** 2)  # Error estándar del lift (incertidumbre del lift): Aproximación (delta method) de la incertidumbre del lift. Construir IC del lift y calcular z-score del lift.
      ci_lift_low = lift - zcrit * se_lift  # Intervalo de confianza del lift
      ci_lift_high = lift + zcrit * se_lift

      if se_lift != 0:
        z_lift = lift/se_lift  # Z-score del lift

      else:
        z_lift = float("nan")

      p_value = p_value_from_z(z_lift)  # p valor
      confidence = 1.0 - p_value  # Confianza

    mdu_abs, mdu_rel = mdu_from_baseline(p_base=p_c, n_c=int(n_c), n_v=int(n_v), conf_level=conf_level, power=power)

    # Difference with MDU (relativo): lift - MDU_rel
    diff_with_mdu = lift - mdu_rel if (not math.isnan(mdu_rel) and not math.isnan(lift)) else float("nan")

    chi2_stat, chi_p = chi_square_p_value_2x2(x_c, x_v, n_c, n_v)

    if not math.isnan(confidence) and confidence >= conf_level:
        winner = "Variante" if p_v > p_c else ("Control" if p_c > p_v else "Empate")
    else:
        winner = "Sin ganador (no significativo)"

    # Summary
    summary_rows.append({
        "test_id": test_id,
        "source": source,
        "definition": definition,
        "from_to": from_to,
        "role": role,
        "metric_name": metric_name,
        "variant_id": row.get("variant_id"),
        "n_control": n_c,
        "x_control": x_c,
        "rate_control": p_c,
        "n_variant": n_v,
        "x_variant": x_v,
        "rate_variant": p_v,
        "lift": lift,
        "ci_lift_low": ci_lift_low,
        "ci_lift_high": ci_lift_high,
        "z_lift": z_lift,
        "p_value": p_value,
        "confidence": confidence,
        "conf_level": conf_level,
        "power": power,
        "mdu_abs": mdu_abs,
        "mdu_rel": mdu_rel,
        "diff_with_mdu": diff_with_mdu,
        "mde_expected": mde_expected,
        #"chi2_stat": chi2_stat,
        #"chi_p_value": chi_p,
        "winner": winner,
        "error": None
    })

    detail_rows.extend([
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Trials", "control": n_c, "variant": n_v},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Conversiones", "control": x_c, "variant": x_v},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Rate", "control": p_c, "variant": p_v},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": f"CI rate low ({int(conf_level*100)}%)", "control": ci_c_low, "variant": ci_v_low},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": f"CI rate high ({int(conf_level*100)}%)", "control": ci_c_high, "variant": ci_v_high},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Lift", "control": None, "variant": lift},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": f"CI lift low ({int(conf_level*100)}%)", "control": None, "variant": ci_lift_low},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": f"CI lift high ({int(conf_level*100)}%)", "control": None, "variant": ci_lift_high},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Z-score (lift)", "control": None, "variant": z_lift},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "p-value (lift)", "control": None, "variant": p_value},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Confianza (1-p)", "control": None, "variant": confidence},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "MDU abs", "control": None, "variant": mdu_abs},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "MDU rel", "control": None, "variant": mdu_rel},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Diff with MDU (rel)", "control": None, "variant": diff_with_mdu},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "MDE esperado", "control": None, "variant": mde_expected},
        #{"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Chi2 stat (2x2)", "control": None, "variant": chi2_stat},
        #{"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Chi-square p-value (2x2)", "control": None, "variant": chi_p},
        {"test_id": test_id, "role": role, "metric_name": metric_name, "metric": "Ganador", "control": None, "variant": winner},
    ])

  df_summary = pd.DataFrame(summary_rows)
  df_detail = pd.DataFrame(detail_rows)
  return df_summary, df_detail

In [ ]:
def state_to_df(state_tests):
  if not state_tests:
      return pd.DataFrame(columns=[
          "test_id",
          "source",
          "definition",
          "from_to",
          "n_control",
          "x_control",
          "n_variant",
          "x_variant",
          "conf_level",
          "power",
          "mde_expected",
          ""
      ])
  return pd.DataFrame(state_tests)

In [ ]:
METRIC_COLS = ["role", "metric_name", "variant_id", "mde_expected", "n_control", "x_control", "n_variant", "x_variant"]

def metrics_state_to_df(state_metrics):
    if not state_metrics:
        return pd.DataFrame(columns=METRIC_COLS)
    return pd.DataFrame(state_metrics)[METRIC_COLS]

def remove_metric(state_metrics, remove_index):
    if not state_metrics:
        return [], metrics_state_to_df_structured([])
    idx = int(float(remove_index or 0))
    if 0 <= idx < len(state_metrics):
        state_metrics.pop(idx)
    return state_metrics, metrics_state_to_df_structured(state_metrics)

def validate_metrics_df(df_metrics: pd.DataFrame):
    if df_metrics is None or df_metrics.empty:
        return False, "No hay métricas. Añade al menos una."

    required_cols = {
        "role", "metric_name", "variant_id",
        "n_control", "x_control", "n_variant", "x_variant"
    }

    missing = required_cols - set(df_metrics.columns)
    if missing:
        return False, f"Faltan columnas obligatorias: {sorted(missing)}"

    # Roles válidos
    valid_roles = {"HEM", "Success", "Proxy", "Guardrail"}
    bad_roles = df_metrics[~df_metrics["role"].astype(str).str.strip().isin(valid_roles)]
    if not bad_roles.empty:
        return False, f"Roles inválidos en filas: {bad_roles.index.tolist()}"

    # Nombre métrica no vacío
    bad_names = df_metrics["metric_name"].astype(str).str.strip().eq("")
    if bad_names.any():
        return False, f"Hay métricas sin nombre en filas: {df_metrics.index[bad_names].tolist()}"

    # Variant ID no vacío
    bad_vid = df_metrics["variant_id"].astype(str).str.strip().eq("")
    if bad_vid.any():
        return False, f"Hay filas sin ID de variante en filas: {df_metrics.index[bad_vid].tolist()}"

    # EXACTAMENTE 1 HEM
    hem_metric_names = (
        df_metrics[df_metrics["role"] == "HEM"]["metric_name"]
        .astype(str).str.strip().unique()
    )

    if len(hem_metric_names) == 0:
        return False, "Debe existir exactamente 1 métrica HEM."

    if len(hem_metric_names) > 1:
        return False, "Solo puede existir una única HEM."

    # Validación n/x
    for i, r in df_metrics.iterrows():
        try:
            n_c = float(r["n_control"])
            x_c = float(r["x_control"])
            n_v = float(r["n_variant"])
            x_v = float(r["x_variant"])
        except Exception:
            return False, f"Fila {i}: valores n/x no numéricos."

        if n_c <= 0 or n_v <= 0:
            return False, f"Fila {i}: n_control y n_variant deben ser > 0."

        if x_c < 0 or x_v < 0:
            return False, f"Fila {i}: conversiones no pueden ser negativas."

        if x_c > n_c or x_v > n_v:
            return False, f"Fila {i}: conversiones fuera de rango (0 ≤ x ≤ n)."

    # Consistencia por métrica
    # (todas las filas de una misma métrica deben compartir mismo control)
    grouped = df_metrics.groupby("metric_name")

    for name, g in grouped:
        if g["n_control"].nunique() > 1 or g["x_control"].nunique() > 1:
            return False, f"La métrica '{name}' tiene distintos valores de control entre variantes."

    return True, "OK"

In [ ]:
# Etiquetas SOLO para la UI (no cambian las columnas internas)
UI_COLS = {
    "test_id": "ID",
    "source": "Fuente",
    "definition": "Definición",
    "from_to": "Desde/Hasta",
    "n_control": "Tamaño de la muestra del control",
    "x_control": "Conversiones del control",
    "n_variant": "Tamaño de la muestra de la variante",
    "x_variant": "Conversiones de la variante",
    "conf_level": "Nivel de confianza",
    "power": "Poder estadístico",
    "rate_control": "Tasa de conversión del control",
    "rate_variant": "Tasa de conversión de la variante",
    "lift": "Lift relativo",
    "ci_lift_low": "IC del lift bajo",
    "ci_lift_high": "IC del lift alto",
    "z_lift": "Z-score del lift",
    "p_value": "p-value del lift",
    "confidence": "Confianza",
    "mdu_abs": "MDU absoluto",
    "mdu_rel": "MDU relativo",
    "diff_with_mdu": "Diferencia MDU",
    "mdu_expected": "MDU esperado",
    "chi2_stat": "Chi2 stat (2x2)",
    "chi_p_value": "p-value Chi2",
    "winner": "Ganador"
}

HELP_KEYS = [
    "n_control", "x_control", "n_variant", "x_variant", "conf_level", "power",
    "rate_control", "rate_variant", "lift", "z_lift", "p_value", "confidence",
    "chi2_stat", "chi_p_value", "mdu_abs", "mdu_rel", "diff_with_mdu", "winner"
]

UI_LABEL_TO_KEY = {v: k for k, v in UI_COLS.items()}

DESC_KEY_MAP = {
    "n_control": "n_c",
    "x_control": "x_c",
    "n_variant": "n_v",
    "x_variant": "x_v",
    "conf_level": "conf",
    "power": "power",

    "rate_control": "rate_c",
    "rate_variant": "rate_v",

    "chi2_stat": "chi2",
    "chi_p_value": "chi2_p",

    "diff_with_mdu": "diff_mdu",
    "mdu_abs": "mdu",
    "mdu_rel": "mdu",
    "winner": "winner",
}

PCT_COLS = {
    "rate_control",
    "rate_variant",
    "lift",
    "confidence",
    "mdu_abs",
    "mdu_rel",
    "diff_with_mdu",
}

def rename_with_ui_cols(df: pd.DataFrame, ui_cols: dict) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    rename_map = {c: ui_cols[c] for c in df.columns if c in ui_cols}
    return df.rename(columns=rename_map)

def format_percentages_for_ui(df: pd.DataFrame, pct_cols: set, decimals: int = 2) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    df_ui = df.copy()

    for c in pct_cols:
        if c in df_ui.columns:
            df_ui[c] = df_ui[c].apply(
                lambda x: f"{x*100:.{decimals}f}%" if pd.notna(x) else x
            )

    return df_ui


In [ ]:
def clear_metrics():
    state = []
    return state, metrics_state_to_df_structured(state)

In [ ]:
def multi_variant_winner_html(rows_df):
    if rows_df is None or rows_df.empty:
        return "<span style='color: orange; font-weight: bold;'>⚠ Sin resultados</span>"

    df = rows_df.copy()

    # Normaliza winner
    df["winner_norm"] = df["winner"].astype(str).str.strip()

    # Ganadoras
    winners = df[df["winner_norm"] == "Variante"].copy()

    # Elegir la de "mayor IC" entre las ganadoras (usamos ci_lift_high)
    best_vid = None
    if not winners.empty and "ci_lift_high" in winners.columns:
        winners["ci_lift_high"] = pd.to_numeric(winners["ci_lift_high"], errors="coerce")
        best_row = winners.sort_values("ci_lift_high", ascending=False).iloc[0]
        best_vid = best_row.get("variant_id", None)

    lines = []
    # Orden por variant_id para que salga A, B, C...
    df_sorted = df.sort_values(by=["variant_id"], kind="stable")

    for _, r in df_sorted.iterrows():
        vid = r.get("variant_id", "")
        w = r.get("winner_norm", "")

        if w == "Variante":
            # La mejor (mayor IC) en verde + etiqueta
            if best_vid is not None and str(vid) == str(best_vid):
                lines.append(
                    f"<span style='color: green; font-weight: bold;'>"
                    f"✔ GANA VARIANTE {vid} (mayor IC)"
                    f"</span>"
                )
            else:
                # Otras ganadoras en azul
                lines.append(
                    f"<span style='color: dodgerblue; font-weight: bold;'>"
                    f"✔ GANA VARIANTE {vid}"
                    f"</span>"
                )

        elif w == "Control":
            # Pierde (control gana)
            lines.append(
                f"<span style='color: red; font-weight: bold;'>"
                f"✘ PIERDE VARIANTE {vid}"
                f"</span>"
            )

        else:
            # Sin ganador / no significativo / etc.
            lines.append(
                f"<span style='color: orange; font-weight: bold;'>"
                f"⚠ VARIANTE {vid}: NO CONCLUYENTE"
                f"</span>"
            )

    return "<br>".join(lines)

In [ ]:
def winner_to_colored_text(winner):
    if winner == "Variante":
        return "<span style='color: green; font-weight: bold;'>✔ GANA LA VARIANTE</span>"
    elif winner == "Control":
        return "<span style='color: red; font-weight: bold;'>✘ GANA EL CONTROL</span>"
    else:
        return "<span style='color: orange; font-weight: bold;'>⚠ SIN GANADOR (no significativo)</span>"


In [ ]:
def plot_test_from_summary_row(row, conf=None):
    # Para que en Colab/Jupyter no se acumulen figuras en salida
    plt.close("all")

    # Extrae inputs (tanto si vienen con nombres internos como ya renombrados)
    n_c = float(row.get("n_control", row.get("n_c", 0)) or 0)
    x_c = float(row.get("x_control", row.get("x_c", 0)) or 0)
    n_v = float(row.get("n_variant", row.get("n_v", 0)) or 0)
    x_v = float(row.get("x_variant", row.get("x_v", 0)) or 0)

    # Nivel de confianza
    if conf is None:
        conf = row.get("conf_level", None)
    conf = float(conf) if conf is not None else 0.95

    # Seguridad
    if n_c <= 0 or n_v <= 0:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.set_title("Rates con IC")
        ax.text(0.5, 0.5, "n_control/n_variant deben ser > 0", ha="center", va="center")
        ax.axis("off")
        return fig, None

    # Usa wilson_ci si existe; si no, usa aproximación normal
    def _wilson_or_normal(x, n, conf_level):
        # Si ya tienes wilson_ci definido en tu notebook, úsalo
        if "wilson_ci" in globals():
            p, lo, hi = wilson_ci(int(x), int(n), conf=conf_level)
            return float(p), float(lo), float(hi)

        # fallback: normal approx
        p = x / n
        # z crítico aprox (solo fallback simple)
        # si tienes z_critical() en tu notebook, úsalo
        if "z_critical" in globals():
            z = float(z_critical(conf_level))
        else:
            z = 1.96 if abs(conf_level - 0.95) < 1e-9 else 1.645

        se = math.sqrt(p * (1 - p) / n) if n > 0 else float("nan")
        lo = max(0.0, p - z * se)
        hi = min(1.0, p + z * se)
        return float(p), float(lo), float(hi)

    p_c, lo_c, hi_c = _wilson_or_normal(x_c, n_c, conf)
    p_v, lo_v, hi_v = _wilson_or_normal(x_v, n_v, conf)

    labels = ["Control", f"Variante {row.get('variant_id','')}".strip() or "Variante"]
    rates = [p_c, p_v]
    lo = [lo_c, lo_v]
    hi = [hi_c, hi_v]

    fig, ax = plt.subplots(figsize=(7, 4))
    x_pos = [0, 1]
    yerr = [
        [rates[i] - lo[i] for i in range(2)],
        [hi[i] - rates[i] for i in range(2)],
    ]

    ax.errorbar(x_pos, rates, yerr=yerr, fmt="o", capsize=5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Rate")
    ax.set_title(f"HEM – Rates con IC {int(round(conf*100))}%")

    # anotaciones
    ax.annotate(f"n={int(n_c)}\nx={int(x_c)}", (0, rates[0]), textcoords="offset points", xytext=(0, 8), ha="center")
    ax.annotate(f"n={int(n_v)}\nx={int(x_v)}", (1, rates[1]), textcoords="offset points", xytext=(0, 8), ha="center")

    fig.tight_layout()
    return fig, None


In [ ]:
def run_compute_direct_multi(test_id, source, definition, from_to, conf, power, state_metrics):
    tid = (test_id or "").strip() or "test_1"

    empty_summary = pd.DataFrame()
    empty_detail = pd.DataFrame()
    empty_last = {"summary": empty_summary, "detail": empty_detail}

    df_metrics = metrics_state_to_df(state_metrics)

    ok, msg = validate_metrics_df(df_metrics)
    if not ok:
        return (
            empty_summary,  # df_summary_view
            empty_detail,   # df_detail_view
            f"❌ {msg}",     # status
            winner_to_colored_text("Sin ganador (no significativo)"),  # winner_md
            empty_last,     # last_results
            None,           # plot_rates
            msg,            # kpi_status
            None,           # download_rates
        )

    try:
        conf = float(conf)
        power = float(power)
    except Exception:
        return (
            empty_summary,
            empty_detail,
            "❌ conf/power inválidos",
            winner_to_colored_text("Sin ganador (no significativo)"),
            empty_last,
            None,
            "Revisa conf/power",
            None,
        )

    # DF para compute: una fila por métrica (y por variante si aplica)
    df_tests = df_metrics.copy()
    df_tests["test_id"] = tid
    df_tests["source"] = source
    df_tests["definition"] = definition
    df_tests["from_to"] = from_to
    df_tests["conf_level"] = conf
    df_tests["power"] = power

    # Cálculo por fila (si compute falla, devolvemos error sin romper UI)
    try:
        df_summary, df_detail = compute(df_tests)
    except Exception as e:
        err = f"❌ Error en compute(): {type(e).__name__}: {e}"
        return (
            empty_summary,
            empty_detail,
            err,
            winner_to_colored_text("Sin ganador (no significativo)"),
            empty_last,
            None,
            err,
            None,
        )

    status = f"✅ Calculadas {len(df_summary)} métricas"
    last = {"summary": df_summary, "detail": df_detail}

    winner_text = winner_to_colored_text("Sin ganador (no significativo)")
    fig_rates = None
    kpi_status = "⚠ No hay resultados para mostrar."

    if df_summary is not None and (not df_summary.empty) and ("role" in df_summary.columns):

        # Selección automática HEM
        hem_rows = df_summary[
            df_summary["role"].astype(str).str.upper().str.strip() == "HEM"
        ]

        if hem_rows.empty:
            winner_text = winner_to_colored_text("Sin ganador (no significativo)")
            fig_rates = None
            kpi_status = "⚠ No se encontró HEM en resultados."

        else:
           # Texto multi-variante (todas las variantes)
          winner_text = multi_variant_winner_html(hem_rows)

          # Para la gráfica: si hay ganadoras, mostrar la de mayor IC (ci_lift_high);
          # si no, mostrar la de mayor IC igualmente (o la primera si falta la columna)
          best = hem_rows.copy()
          if "ci_lift_high" in best.columns:
              best["ci_lift_high"] = pd.to_numeric(best["ci_lift_high"], errors="coerce")
              best = best.sort_values("ci_lift_high", ascending=False).iloc[0]
          else:
              best = best.iloc[0]

          vid = best.get("variant_id", "")
          pval = best.get("p_value", None)

          fig_rates, _ = plot_test_from_summary_row(best)

          pval = best.get("p_value", None)
          hem_name = best.get("metric_name", "")

          kpi_status = f"HEM: **{hem_name}** | Variante mostrada (mayor IC): {vid}"
          if pval is not None and pd.notna(pval):
              kpi_status += f" | p={float(pval):.6f}"

    # Formateo UI (p_value en decimal + % en tasas)
    df_summary_ui = df_summary.copy() if df_summary is not None else empty_summary.copy()

    if "p_value" in df_summary_ui.columns:
        df_summary_ui["p_value"] = df_summary_ui["p_value"].apply(
            lambda p: f"{float(p):.12f}" if (p is not None and pd.notna(p)) else p
        )

    df_summary_ui = rename_with_ui_cols(
        format_percentages_for_ui(df_summary_ui, PCT_COLS),
        UI_COLS
    )

    # Descarga del plot (rates) sin romper si no hay figura
    rates_path = None
    try:
        if fig_rates is not None:
            rates_path = save_figure_to_png(fig_rates, prefix=f"{tid}_rates_ci")
    except Exception as e:
        print("Error guardando figura:", e)
        rates_path = None

    return (
        df_summary_ui,
        df_detail,
        status,
        winner_text,
        last,
        fig_rates,
        kpi_status,
        rates_path,
    )



In [ ]:
def export_results_to_csv(last_results_state):

    df_summary = last_results_state.get("summary", pd.DataFrame())
    df_detail  = last_results_state.get("detail", pd.DataFrame())

    if df_summary is not None and not df_summary.empty:
        df = df_summary.copy()

        # Filtra por roles válidos (métricas reales)
        if "role" in df.columns:
            df["role"] = df["role"].astype(str).str.strip()
            df = df[df["role"].isin(["HEM", "Success", "Proxy", "Guardrail"])]

        # Asegura metric_name no vacío
        if "metric_name" in df.columns:
            df["metric_name"] = df["metric_name"].astype(str).str.strip()
            df = df[df["metric_name"] != ""]

        df_summary_csv = df.reset_index(drop=True)
    else:
        df_summary_csv = pd.DataFrame()

    with tempfile.NamedTemporaryFile(delete=False, suffix=".csv", mode="w", encoding="utf-8") as f:
        if not df_summary_csv.empty:
            df_summary_csv.to_csv(f, index=False)
        else:
            f.write("No hay métricas para exportar.\n")
        return f.name


def export_click(last_results_state):
    path = export_results_to_csv(last_results_state)
    return path


In [ ]:
def wilson_ci(x, n, conf=0.95):
    # IC Wilson para proporciones
    if n <= 0:
        return (float("nan"), float("nan"), float("nan"))
    p = x / n
    # z approx (sin scipy)
    # valores comunes; si usas conf slider 0.80-0.99 esto cubre bien.
    z_table = {0.80:1.2816, 0.85:1.4395, 0.90:1.6449, 0.95:1.96, 0.98:2.3263, 0.99:2.5758}
    z = z_table.get(round(conf, 2), 1.96)

    denom = 1 + (z*z)/n
    center = (p + (z*z)/(2*n)) / denom
    half = (z * math.sqrt((p*(1-p) + (z*z)/(4*n)) / n)) / denom
    lo = max(0.0, center - half)
    hi = min(1.0, center + half)
    return (p, lo, hi)

In [ ]:
def make_kpi_panel_from_summary_row(row: pd.Series) -> pd.DataFrame:

    def fmt_num(x, nd=6):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return ""
        try:
            return f"{float(x):.{nd}g}"
        except Exception:
            return str(x)

    panel = pd.DataFrame({
        "KPI": [
            "test_id",
            "winner",
            "p-value (lift)",
            "confianza (1-p)",
            "chi-square p-value",
            "n_control",
            "x_control",
            "rate_control",
            "n_variant",
            "x_variant",
            "rate_variant",
        ],
        "Valor": [
            str(row.get("test_id", "")),
            str(row.get("winner", "")),
            fmt_num(row.get("p_value")),
            fmt_num(row.get("confidence")),
            fmt_num(row.get("chi_p_value")),
            str(int(row.get("n_control"))) if pd.notna(row.get("n_control")) else "",
            str(int(row.get("x_control"))) if pd.notna(row.get("x_control")) else "",
            fmt_num(row.get("rate_control")),
            str(int(row.get("n_variant"))) if pd.notna(row.get("n_variant")) else "",
            str(int(row.get("x_variant"))) if pd.notna(row.get("x_variant")) else "",
            fmt_num(row.get("rate_variant")),
        ]
    })
    return panel

In [ ]:
def show_desc(raw_key):
    k = (raw_key or "").strip()

    # si llega el label de UI, conviértelo a key interna
    if k in UI_LABEL_TO_KEY:
        k = UI_LABEL_TO_KEY[k]

    # adapta a la key que usa descriptions
    k2 = DESC_KEY_MAP.get(k, k)

    # busca descripción
    desc = descriptions.get(k2)
    if desc:
        return desc

    return f"❌ No hay descripción para este parámetro.\n\n- recibido: `{raw_key}`\n- key interna: `{k}`\n- key en descriptions: `{k2}`"


In [ ]:
def save_figure_to_png(fig, prefix="plot"):
    if fig is None:
        return None

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".png", prefix=f"{prefix}_")
    fig.savefig(tmp.name, bbox_inches="tight", dpi=150)
    tmp.close()
    return tmp.name

In [ ]:
descriptions = {

    # =====================
    # PARÁMETROS DE ENTRADA
    # =====================

    "n_c": """**Tamaño de la muestra de control (n_control):**
Número total de usuarios, sesiones u observaciones asignadas al grupo Control.
Define el tamaño de la muestra sobre la que se calcula la tasa de conversión del control.
""",

    "x_c": """**Número de conversiones en la muestra de control (x_control):**
Cantidad de usuarios del grupo Control que realizaron la conversión definida.
Debe cumplir: 0 ≤ x_control ≤ n_control.
""",

    "n_v": """**Tamaño de la muestra de variante (n_variant):**
Número total de usuarios, sesiones u observaciones asignadas al grupo Variante.
Define el tamaño de la muestra sobre la que se calcula la tasa de conversión de la variante.
""",

    "x_v": """**Número de conversiones en la muestra de variante (x_variant):**
Cantidad de usuarios del grupo Variante que realizaron la conversión definida.
Debe cumplir: 0 ≤ x_variant ≤ n_variant.
""",

    "conf": """**Nivel de confianza (conf_level):**
Probabilidad complementaria al p-value utilizada para decidir significación estadística.
Por ejemplo, un nivel de confianza del 95% equivale a un nivel de significación α = 0.05.
Se usa para:
- calcular los intervalos de confianza
- decidir el ganador del test
""",

    "power": """**Potencia estadística (power):**
Probabilidad de detectar un efecto real de tamaño MDU o mayor si dicho efecto existe.
Se utiliza exclusivamente para el cálculo del MDU.
""",

    "mde": """**MDE esperado (mde_expected):**
Es el cambio mínimo en la métrica (normalmente expresado en % de lift) que esperamos o queremos ser capaces de detectar como estadísticamente significativo en el experimento.
""",

    "num_variants": """**Númeo de variantes:**
Aquí se puede introducir el número de variables que deseas para el test.
""",

    # =====================
    # MÉTRICAS BÁSICAS
    # =====================

    "rate_c": """**Tasa de conversión del control (rate_control):**
Proporción de conversiones en el grupo Control.
Se calcula como: x_control / n_control.
""",

    "rate_v": """**Tasa de conversión de la variante (rate_variant):**
Proporción de conversiones en el grupo Variante.
Se calcula como: x_variant / n_variant.
""",

    # =====================
    # INCERTIDUMBRE E IC
    # =====================

    "se_c": """**Error estándar del rate de control:**
Medida de la variabilidad esperada de la tasa de conversión del Control debido al muestreo.
Se calcula asumiendo un modelo binomial.
""",

    "se_v": """**Error estándar del rate de variante:**
Medida de la variabilidad esperada de la tasa de conversión de la Variante debido al muestreo.
Se calcula asumiendo un modelo binomial.
""",

    "ci_rate_c": """**Intervalo de confianza del rate de control:**
Rango de valores plausibles para la tasa real de conversión del Control al nivel de confianza seleccionado.
Se calcula como: rate_control ± z · SE_control.
""",

    "ci_rate_v": """**Intervalo de confianza del rate de variante:**
Rango de valores plausibles para la tasa real de conversión de la Variante al nivel de confianza seleccionado.
Se calcula como: rate_variant ± z · SE_variant.
""",

    # =====================
    # EFECTO (LIFT)
    # =====================

    "lift": """**Lift (uplift relativo):**
Cambio relativo de la tasa de conversión de la Variante respecto al Control.
Se calcula como: (rate_variant − rate_control) / rate_control.
Un lift positivo indica mejora; negativo indica empeoramiento.
""",

    "se_lift": """**Error estándar del lift:**
Estimación de la incertidumbre asociada al lift, calculada mediante el método delta.
Se utiliza para construir intervalos de confianza y el estadístico Z del lift.
""",

    "ci_lift": """**Intervalo de confianza del lift:**
Rango de valores plausibles para el lift real.
Si el intervalo no contiene el valor 0, el efecto se considera estadísticamente significativo.
""",

    # =====================
    # SIGNIFICACIÓN
    # =====================

    "z_lift": """**Z-score del lift:**
Número de errores estándar que separan el lift observado del valor nulo (0).
Bajo la hipótesis nula, sigue aproximadamente una distribución normal estándar.
""",

    "p_value": """**p-value del test:**
Probabilidad de observar un efecto igual o más extremo que el observado si no existiera ningún efecto real.
Un p-value pequeño indica evidencia contra la hipótesis nula.
""",

    "confidence": """**Confianza estadística (1 − p-value):**
Evidencia (1−p) (derivada del p-value):
Esta cifra indica la evidencia contra la hipótesis nula H0 según el test estadístico, y se compara con el nivel de confianza configurado (conf_level).
No representa “certeza absoluta” ni “probabilidad real de que la variante gane”.
Si el p-value es extremadamente pequeño, 1−p puede redondear a “100%”.
""",

    # =====================
    # POTENCIA Y MDU
    # =====================

    "mdu": """**MDU – Minimum Detectable Uplift:**
Mínimo cambio relativo que el experimento puede detectar con la potencia y nivel de confianza definidos.
Depende del tamaño de muestra, la tasa base y la potencia estadística.
""",

    "diff_mdu": """**Difference with MDU:**
Diferencia entre el lift observado y el MDU.
Valores negativos indican que el experimento no tenía potencia suficiente para detectar el efecto observado.
""",

    # =====================
    # CHI-SQUARE
    # =====================

    "chi2": """**Estadístico Chi-cuadrado (2×2):**
Medida de discrepancia entre las conversiones observadas y las esperadas bajo la hipótesis de independencia.
Se calcula sobre la tabla Control/Variante × Conversión/No conversión.
""",

    "chi2_p": """**p-value del test Chi-cuadrado:**
Probabilidad de observar una discrepancia igual o mayor si la conversión fuera independiente del grupo.
Sirve como validación alternativa al test Z.
""",

    # =====================
    # DECISIÓN
    # =====================

    "winner": """**Ganador del experimento:**
Resultado final del test según la regla de decisión estadística.
- Variante: mejora significativa
- Control: empeoramiento significativo
- Sin ganador: diferencia no significativa
"""
}


In [ ]:
def _z_from_conf(conf: float) -> float:
    # aproximación sin scipy (suficiente para 0.80-0.99)
    table = {0.80:1.2816, 0.85:1.4395, 0.90:1.6449, 0.95:1.96, 0.98:2.3263, 0.99:2.5758}
    return table.get(round(float(conf), 2), 1.96)

def wilson_ci(x, n, conf=0.95):
    if n <= 0:
        return (float("nan"), float("nan"), float("nan"))
    p = x / n
    z = _z_from_conf(conf)
    denom = 1 + (z*z)/n
    center = (p + (z*z)/(2*n)) / denom
    half = (z * math.sqrt((p*(1-p) + (z*z)/(4*n)) / n)) / denom
    lo = max(0.0, center - half)
    hi = min(1.0, center + half)
    return p, lo, hi

def build_rates_plot_all_variants(state_metrics, conf=0.95):
    # Si no hay métricas
    if not state_metrics:
        fig, ax = plt.subplots()
        ax.set_title("Rates con IC")
        ax.text(0.5, 0.5, "No hay métricas", ha="center", va="center")
        ax.axis("off")
        return fig, None, None

    metrics = state_metrics
    fig_h = max(3.5, 2.6 * len(metrics))
    fig, axes = plt.subplots(len(metrics), 1, figsize=(10, fig_h), squeeze=False)
    axes = axes.flatten()

    for ax, m in zip(axes, metrics):
        metric_name = m.get("metric_name", "") or "Métrica"
        ctrl = m.get("control", {}) or {}
        vars_ = m.get("variants", []) or []

        # construye lista: Control primero, luego variantes
        groups = [("C", ctrl.get("n", 0), ctrl.get("x", 0))]
        for v in vars_:
            groups.append((str(v.get("variant_id","")), v.get("n", 0), v.get("x", 0)))

        # ordena variantes por id pero mantiene C primero
        c = groups[0]
        rest = sorted(groups[1:], key=lambda t: t[0])
        groups = [c] + rest

        labels = [g[0] for g in groups]
        rates, lo, hi = [], [], []
        for _, n, x in groups:
            p, l, h = wilson_ci(int(x), int(n), conf=conf)
            rates.append(p); lo.append(l); hi.append(h)

        x_pos = list(range(len(labels)))
        yerr = [[r - l for r, l in zip(rates, lo)],
                [h - r for h, r in zip(hi, rates)]]

        ax.errorbar(x_pos, rates, yerr=yerr, fmt="o", capsize=4)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Rate")
        ax.set_title(metric_name)

        # anotaciones n/x
        for i, (_, n, x) in enumerate(groups):
            ax.annotate(f"n={int(n)}\nx={int(x)}", (x_pos[i], rates[i]),
                        textcoords="offset points", xytext=(0, 8), ha="center")

    fig.suptitle(f"Rates con IC {int(round(conf*100))}%", y=0.995)
    fig.tight_layout()

    tmpdir = tempfile.mkdtemp()
    outpath = os.path.join(tmpdir, "rates_ic.png")
    fig.savefig(outpath, dpi=150, bbox_inches="tight")
    return fig, outpath, None

In [ ]:
MAX_VARIANTS = 5

def _to_int(v, default=0):
    if v in (None, ""):
        return default
    return int(float(v))  # gr.Number suele venir como float

def _variant_labels(k, base="A"):
    # Por defecto: A, B, C... si base es una letra; si no, usa V1..Vk
    if isinstance(base, str) and len(base.strip()) == 1 and base.strip().isalpha():
        start = ord(base.strip().upper())
        return [chr(start + i) for i in range(k)]
    return [f"V{i+1}" for i in range(k)]

def set_variant_visibility(k):
    """k = número de variantes (sin contar control). Devuelve updates para cada Group de variante."""
    k = int(k or 0)
    updates = []
    for i in range(MAX_VARIANTS):
        updates.append(gr.update(visible=(i < k)))
    return updates

def metrics_state_to_df_structured(state_metrics):
    rows = []
    for i, m in enumerate(state_metrics or []):
        role = m.get("role", "")
        metric_name = m.get("metric_name", "")
        mde = m.get("mde_expected", None)
        ctrl = m.get("control", {}) or {}
        variants = m.get("variants", []) or []

        # una fila por variante para que la tabla se entienda
        if not variants:
            rows.append({
                "idx": i,
                "role": role,
                "metric_name": metric_name,
                "variant_id": None,
                "mde_expected": mde,
                "n_control": ctrl.get("n"),
                "x_control": ctrl.get("x"),
                "n_variant": None,
                "x_variant": None,
            })
        else:
            for v in variants:
                rows.append({
                    "idx": i,
                    "role": role,
                    "metric_name": metric_name,
                    "variant_id": v.get("variant_id"),
                    "mde_expected": mde,
                    "n_control": ctrl.get("n"),
                    "x_control": ctrl.get("x"),
                    "n_variant": v.get("n"),
                    "x_variant": v.get("x"),
                })

    return pd.DataFrame(rows)

def add_metric_structured(
    state_metrics, role, metric_name, mde_expected_pct,
    n_c, x_c, num_variants, *variant_vals
):
    if state_metrics is None:
        state_metrics = []

    def _to_int(v, default=0):
        if v in (None, ""):
            return default
        try:
            return int(float(v))
        except Exception:
            return default

    def _to_k(v, default=0):
        if v in (None, ""):
            return default
        try:
            k = int(float(v))
            return max(0, k)
        except Exception:
            return default

    def _clamp_counts(n, x):
        # n >= 0, x in [0, n]
        n = max(0, int(n))
        x = max(0, int(x))
        if x > n:
            x = n
        return n, x

    def _variant_labels(k, base="A"):
        base = (base or "A").strip()
        if len(base) == 1 and base.isalpha():
            start = ord(base.upper())
            return [chr(start + i) for i in range(k)]
        return [f"V{i+1}" for i in range(k)]

    # --- Limpieza inputs básicos ---
    role = (role or "").strip()
    metric_name = (metric_name or "").strip()

    # --- MDE esperado ---
    mde_pct = 0.0 if mde_expected_pct in (None, "") else float(mde_expected_pct)
    mde = mde_pct / 100.0

    # --- Nº de variantes ---
    k = _to_k(num_variants, default=0)

    # --- Control ---
    n_c_i = _to_int(n_c, default=0)
    x_c_i = _to_int(x_c, default=0)
    n_c_i, x_c_i = _clamp_counts(n_c_i, x_c_i)

    # --- Variantes: coger exactamente 2*k valores, rellenando con None si faltan ---
    vals = list(variant_vals)
    needed = 2 * k
    if len(vals) < needed:
        vals = vals + [None] * (needed - len(vals))
    else:
        vals = vals[:needed]

    labels = _variant_labels(k, base="A")

    variants = []
    for i in range(k):

        n_i = _to_int(vals[2 * i], default=0)
        x_i = _to_int(vals[2 * i + 1], default=0)
        n_i, x_i = _clamp_counts(n_i, x_i)

        variants.append({
            "variant_id": labels[i],   # A, B, C...
            "n": n_i,
            "x": x_i,
        })

    row = {
        "role": role,
        "metric_name": metric_name,
        "mde_expected": mde,
        "control": {"variant_id": "C", "n": n_c_i, "x": x_c_i},
        "variants": variants,
        "k_variants": k,
    }

    state_metrics.append(row)
    return state_metrics, metrics_state_to_df_structured(state_metrics)


def flatten_structured_metrics_for_compute(state_metrics):
    rows = []

    for metric in state_metrics:
        for v in metric["variants"]:
            rows.append({
                "role": metric["role"],
                "metric_name": metric["metric_name"],
                "variant_id": v["variant_id"],
                "mde_expected": metric["mde_expected"],
                "n_control": metric["control"]["n"],
                "x_control": metric["control"]["x"],
                "n_variant": v["n"],
                "x_variant": v["x"]
            })

    return rows



def run_compute_direct_multi_structured(test_id, source, definition, from_to, conf, power, state_metrics):

    plt.close("all")
    # 1) Aplanar para compute antiguo
    flat_state = flatten_structured_metrics_for_compute(state_metrics)

    result = run_compute_direct_multi(
        test_id, source, definition, from_to, conf, power, flat_state
    )

    # Si run_compute devolvió algo raro, no seguimos
    if not isinstance(result, tuple) or len(result) != 8:
        return result

    df_summary, df_detail, status, winner_md, last_results, plot_rates_old, kpi_status, download_rates_old = result

    # Si hay error en status, no intentamos graficar
    if status.startswith("❌"):
        return df_summary, df_detail, status, winner_md, last_results, None, kpi_status, None

    # 2) Plot nuevo (control + todas variantes)
    try:
        fig, plot_path, _ = build_rates_plot_all_variants(state_metrics, conf=float(conf))
    except Exception as e:
        print("Error en build_rates_plot_all_variants:", e)
        fig = None
        plot_path = None

    return df_summary, df_detail, status, winner_md, last_results, fig, kpi_status, plot_path


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown('# Calculadora de Resultados')

    help_md = gr.Markdown("Selecciona un parámetro y pulsa ℹ️ para ver la descripción.")
    last_results = gr.State({'summary': pd.DataFrame(), 'detail': pd.DataFrame()})
    state_metrics = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            test_id = gr.Textbox(label=UI_COLS.get('test_id','test_id'), placeholder='ID')
            source = gr.Textbox(label=UI_COLS.get('source','source'), placeholder='ej: GA4')
            definition = gr.Textbox(label=UI_COLS.get('definition','definition'), placeholder='ej: Order Rate')
            from_to = gr.Textbox(label=UI_COLS.get('from_to','from_to'), placeholder='ej: PDP -> Purchase')

            gr.Markdown("## Confianza y poder estadístico")
            with gr.Row():
                conf = gr.Slider(0.80, 0.99, value=0.95, step=0.01, label=UI_COLS.get('conf_level','conf_level'))
                btn_conf = gr.Button("ℹ️", scale=0, min_width=40)
            with gr.Row():
                power = gr.Slider(0.80, 0.99, value=0.80, step=0.01, label=UI_COLS.get('power','power'))
                btn_power = gr.Button("ℹ️", scale=0, min_width=40)

            gr.Markdown("## MDE esperado")
            with gr.Row():
                mde_expected = gr.Number(label="MDE esperado (%)", value=0.0, precision=2)
                btn_mde = gr.Button("ℹ️", scale=0, min_width=40)

            gr.Markdown("## Control")
            with gr.Row():
                n_c = gr.Number(label=UI_COLS.get('n_control','n_control'), value=10000, precision=0)
                btn_n_c = gr.Button("ℹ️", scale=0, min_width=40)
            with gr.Row():
                x_c = gr.Number(label=UI_COLS.get('x_control','x_control'), value=400, precision=0)
                btn_x_c = gr.Button("ℹ️", scale=0, min_width=40)

            gr.Markdown("## Variantes")
            with gr.Row():
                num_variants = gr.Slider(1, MAX_VARIANTS, value=1, step=1, label="Número de variantes")
                btn_num_variants = gr.Button("ℹ️", scale=0, min_width=40)


            # Variantes dinámicas: precreadas y se muestran según num_variants
            variant_groups = []
            variant_n_inputs = []
            variant_x_inputs = []

            for i in range(MAX_VARIANTS):
                with gr.Group(visible=(i < 1)) as g:
                    gr.Markdown(f"**Variante {i+1}**")
                    with gr.Row():
                        n_v = gr.Number(label=f"{UI_COLS.get('n_variant','n_variant')} {i+1}", value=10000, precision=0)
                        btn_n_v = gr.Button("ℹ️", scale=0, min_width=40)
                    with gr.Row():
                        x_v = gr.Number(label=f"{UI_COLS.get('x_variant','x_variant')} {i+1}", value=450, precision=0)
                        btn_x_v = gr.Button("ℹ️", scale=0, min_width=40)

                variant_groups.append(g)
                variant_n_inputs.append(n_v)
                variant_x_inputs.append(x_v)

                # tooltips: los conectamos a las mismas keys que ya tenías
                btn_n_v.click(lambda: show_desc("n_v"), outputs=help_md)
                btn_x_v.click(lambda: show_desc("x_v"), outputs=help_md)

            num_variants.change(
                set_variant_visibility,
                inputs=[num_variants],
                outputs=variant_groups
            )

            gr.Markdown("## Métricas del experimento")
            with gr.Row():
                role = gr.Dropdown(choices=["HEM","Success","Proxy","Guardrail"], value="HEM", label="Rol")
                metric_name = gr.Textbox(label="Nombre métrica", placeholder="Ej: Add to cart rate")

            with gr.Row():
                btn_add_metric = gr.Button("➕ Añadir métrica", variant="primary")
            with gr.Row():
                btn_clear_metrics = gr.Button("🧹 Limpiar métricas")

            with gr.Row():
                remove_index = gr.Number(label="Índice a eliminar (0,1,2...)", value=0, precision=0)
                btn_remove_metric = gr.Button("🗑️ Eliminar métrica")

            df_metrics_view = gr.Dataframe(label="Métricas añadidas", interactive=False)

            with gr.Row():
                btn_compute = gr.Button('Calcular', variant="primary")
            with gr.Row():
                btn_export = gr.Button('Exportar CSV')

            status = gr.Textbox(label='Estado', interactive=False)
            download = gr.File(label='Descargar CSV')

        with gr.Column(scale=3):
            winner_md = gr.Markdown()
            df_summary_view = gr.Dataframe(label='Resultados (summary)', interactive=False)
            df_detail_view = gr.Dataframe(label='Resultados', interactive=False)

            gr.Markdown("Gráficas")

            with gr.Row():
                plot_rates = gr.Plot(label="Rates con IC")

            with gr.Row():
                download_rates = gr.File(label="Descargar gráfica Rates (IC)")

            kpi_status = gr.Markdown()

            gr.Markdown("## Ayuda de resultados (calculados)")
            calc_key = gr.Dropdown(
                choices=[(UI_COLS.get(k, k), k) for k in HELP_KEYS],  # (label visible, value interno)
                value="lift",
                label="Parámetro / estadístico"
            )
            btn_calc_help = gr.Button("ℹ️ Ver descripción")
            calc_help_md = gr.Markdown()


    btn_compute.click(
        run_compute_direct_multi_structured,
        inputs=[test_id, source, definition, from_to, conf, power, state_metrics],
        outputs=[df_summary_view, df_detail_view, status, winner_md, last_results, plot_rates, kpi_status, download_rates]
    )

    btn_export.click(
        export_click,
        inputs=[last_results],
        outputs=[download]
    )

    btn_calc_help.click(
        show_desc,
        inputs=[calc_key],
        outputs=[calc_help_md]
    )

    all_variant_inputs = []
    for i in range(MAX_VARIANTS):
        all_variant_inputs += [variant_n_inputs[i], variant_x_inputs[i]]

    btn_add_metric.click(
        add_metric_structured,
        inputs=[state_metrics, role, metric_name, mde_expected, n_c, x_c, num_variants] + all_variant_inputs,
        outputs=[state_metrics, df_metrics_view]
    )

    btn_remove_metric.click(
        remove_metric,
        inputs=[state_metrics, remove_index],
        outputs=[state_metrics, df_metrics_view]
    )

    btn_clear_metrics.click(
        clear_metrics,
        inputs=[],
        outputs=[state_metrics, df_metrics_view]
    )


    # Tooltips ℹ️
    btn_n_c.click(lambda: show_desc("n_c"), outputs=help_md)
    btn_x_c.click(lambda: show_desc("x_c"), outputs=help_md)
    btn_conf.click(lambda: show_desc("conf"), outputs=help_md)
    btn_power.click(lambda: show_desc("power"), outputs=help_md)
    btn_mde.click(lambda: show_desc("mde"), outputs=help_md)
    btn_num_variants.click(lambda: show_desc("num_variants"), outputs=help_md)

if __name__ == '__main__':
    demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0d5adae920ced18713.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 